<div align="center">
    <img src="assets/neovim-logo.svg" width="260" class="no-zoom" alt="Neovim"/>
</div>

# Neovim Essentials: Day 4 - Treesitter and LSPs: In Depth

<div align="center">
<a href="https://www.lazyvim.org/plugins"><img src="https://img.shields.io/badge/Neovim-Text_Objects_%26_Plugins-019733?style=for-the-badge&logo=neovim&logoColor=white" alt="Neovim: Treesitter and LSPs"/></a>
<img src="https://img.shields.io/badge/Day-4-57A143?style=for-the-badge" alt="Day 4"/>
<a href="https://github.com/neovim/neovim/releases"><img src="https://img.shields.io/github/v/release/neovim/neovim?style=for-the-badge&label=latest&color=57A143" alt="Latest Neovim release"/></a>
</div>

On Day 04, we will take a look at a bit of an involved concepts: tree-sitter and LSPs and how they interact with each other.

### 🗺️ Today's roadmap

| # | Section | What you'll get out of it |
|---|---|---|
| 1 | 🎯 Treesitter | What Treesitter is, and why it's built around parsers |
| 2 | 🔤 Lexical analysis | How source code is split into tokens |
| 3 | 🌳 Syntactic analysis | How a parser checks tokens against a grammar and builds a syntax tree |
| 4 | 📜 The WinZig grammar | Reading a real grammar: rules, notation and railroad diagrams |
| 5 | 🔬 Parsing one line | `x := a + b * 2` from tokens to parse tree to AST, compared with Python's |
| 6 | 🧮 Semantic analysis | Symbol tables and type checking, and how language servers use them |
| 7 | ⚙️ Intermediate representation | Turning the AST into simple steps: three-address code and Python bytecode |

## 🎯 Treesitter

Treesitter is an open-source project that combines a list of parser generators. It has a set of generators devoted for each language through which you can input the source code and obtain the **A**bstract **S**yntax **T**ree (AST).

### Parser: What is it?

A parser is literally the second step of a compiler. A compiler, in general, is composed of several step, whose outputs are fundamentally more abstract than that of the previous step's output. Most compilers start with a string of source code (a "source code string" if you will), and come up with an **I**ntermediate **R**epresentation (IR) which is far more abstract than the string of source code. The IR embeds the _syntactic_ and - more importantly - **_semantic_** meaning of your source code.

### Compilers 101

The frontend of a compiler is made of the following steps.

1. Lexical Analysis
2. Syntactic Analysis
3. Semantic Analysis
4. Intermediate Representation

**Lexical Analysis**

The input for this step is the raw source code. The compiler reads it character by character and groups the characters into **tokens**: keywords, identifiers, numbers, operators and so on. Every programming language is defined by a specification, and the specification includes a **grammar** that says what the valid tokens are (used by the lexer) and how they can be combined (used by the parser).

**Syntactic Analysis**

The lexer only checks that each *word* is valid. The **parser** checks that the words are in a valid *order*, by matching the list of tokens against the grammar's rules. Take the line `x := a + b * 2;` from the example program further down, and change it to `x := a + ;`: every token is still valid, but the rule `Term '+' Factor` needs a `Factor` after the `+`, and `;` can't start one, so the parser stops with a **syntax error**.

When the tokens do fit, the parser records which rules it used, as a tree. The first tree is the **parse tree** (or *concrete syntax tree*), which holds every token, including `:=`, `;` and `begin`. From it, the parser builds the smaller **abstract syntax tree (AST)**, which keeps only the structure that matters: in WinZig, the rules labelled `=> "name"`. *Example: parsing one line*, further down, shows both trees for `x := a + b * 2`.

Parsers are either written by hand, typically with one function per grammar rule (a *recursive-descent* parser), or generated from the grammar by a **parser generator** such as Yacc/Bison or ANTLR. Treesitter is a parser generator too: each language's Treesitter parser is generated from a grammar file, and it builds a syntax tree for every buffer you open in Neovim.

**Semantic Analysis**

A program can follow the grammar perfectly and still make no sense, much like the English sentence *"Colourless green ideas sleep furiously"*. The grammar only describes the *shape* of the code, so it can't catch this. Semantic analysis walks the AST and checks its **meaning**:

- **Name resolution**: is every name declared, and which declaration does it refer to? The compiler builds a **symbol table** that records each name, its kind (variable, constant, function, type) and its type. It also tracks **scopes**, so a variable declared inside a function isn't visible outside it.
- **Type checking**: is each operator applied to the right kind of value? `+` needs two integers, and `:=` needs both sides to have the same type.
- **The language's other rules**: a function is called with the right number of arguments, `exit` only appears inside a loop, a constant is never assigned to, and so on.

The example program further down declares `var x, a, b : integer;`, so its symbol table holds:

| Name | Kind | Type |
|---|---|---|
| `x` | variable | `integer` |
| `a` | variable | `integer` |
| `b` | variable | `integer` |

To check `x := a + b * 2`, the checker works up the AST from the leaves and gives each node a type:

```text
assign                       ✓ integer := integer
├── <identifier> x           integer   (from the symbol table)
└── +                        integer   (integer + integer)
    ├── <identifier> a       integer   (from the symbol table)
    └── *                    integer   (integer * integer)
        ├── <identifier> b   integer   (from the symbol table)
        └── <integer> 2      integer   (a literal)
```

Change only the declarations, and the same line, which still parses fine, fails:

| Declarations | Result |
|---|---|
| `var x, a, b : integer;` | ✓ OK |
| `var x, b : integer;` | ✗ `a` is not declared |
| `var x, b : integer; a : char;` | ✗ `+` cannot be applied to a `char` |

The output of this step is the AST annotated with types, plus the symbol table.

> 💡 This is also what a **language server** does as you type. When pyright underlines an undefined variable or a type mismatch, that's semantic analysis, run on your code without compiling or running it. Python itself does most of these checks only when the code runs, because it is dynamically typed.

**Intermediate Representation**

Finally, the checked AST is translated into an **intermediate representation (IR)**: a simpler, lower-level form of the program that no longer looks like the source language. Trees are good for checking code, but awkward to optimise and to turn into machine instructions, so the IR flattens the tree into a list of small, explicit steps. Because it doesn't depend on either the source language or the target processor, one IR can be shared by many languages and many processors.

A common textbook IR is **three-address code**: each instruction does at most one operation, and intermediate results go into temporary variables (`t1`, `t2`, ...). For `x := a + b * 2`:

```text
t1 = b * 2
t2 = a + t1
x  = t2
```

The order of evaluation, which was only implied by the shape of the AST, is now spelled out: `b * 2` is computed first.

Real compilers do the same. Clang, rustc and Swift all translate to **LLVM IR**, and Java compiles to **JVM bytecode**. Python compiles to its own **bytecode**, which you can see with the `dis` module: `python3 -c "import dis; dis.dis(compile('x = a + b * 2', '<line>', 'exec'))"` prints (Python 3.13):

```text
LOAD_NAME                0 (a)
LOAD_NAME                1 (b)
LOAD_CONST               0 (2)
BINARY_OP                5 (*)
BINARY_OP                0 (+)
STORE_NAME               2 (x)
```

Python's bytecode is *stack-based* rather than three-address code: each `LOAD_` instruction pushes a value onto a stack, and each `BINARY_OP` pops two values and pushes the result. The order is the same, though: `b * 2` first, then `a +` that result, then store it in `x`.

The IR is where the frontend ends. From here, the compiler's **backend** optimises the IR and generates machine code for a specific processor.

All four steps are driven by the language's specification, and the lexer and parser by its grammar. Python's grammar is large (see the [lexical analysis](https://docs.python.org/3/reference/lexical_analysis.html) and [full grammar](https://docs.python.org/3/reference/grammar.html) chapters of the language reference), so let's look at a much smaller one instead: **WinZig**, a small Pascal-like teaching language.

<details>
<summary><b>Example: the WinZig language</b></summary>

##### How to read the grammar

Each rule reads `NonTerminal -> what it can be made of`, with one line per alternative:

- Quoted items such as `'begin'` are **terminals**: tokens that must appear literally.
- Unquoted names such as `Body` are **non-terminals**: other rules.
- `<identifier>`, `<integer>`, `<char>` and `<string>` are **token classes**, defined at the top of the grammar.
- An empty right-hand side means the rule can match nothing.
- `=> "name"` names the **AST node** the parser builds when the rule matches. Rules without one (like `'(' Expression ')'`) build no node of their own, which is what makes the tree *abstract*: brackets, semicolons and keywords disappear, and only the structure remains.
- `--` starts a remark that is not part of the grammar.

The grammar also uses a few operators. When they are *not* in quotes, they are part of the notation, not tokens:

| Notation | Meaning | Example | Reads as |
|---|---|---|---|
| `X+` | One or more `X` | `'type' (Type ';')+` | `type`, followed by one or more `Type ;` |
| `X*` | Zero or more `X` | `Fcn*` | Any number of functions, possibly none |
| `X?` | Optional `X` | `('else' Statement)?` | The `else` branch may be left out |
| `X \| Y` | Either `X` or `Y` | `'a' \| 'b'` | `a` or `b` |
| `X list Y` | One or more `X`, separated by `Y`; short for `X (Y X)*` | `Statement list ';'` | Statements separated by `;` (no `;` after the last one) |

##### The grammar

> 📝 **Note:** This is the grammar we used to develop the **WinZigC** compiler in my compiler class.

```text
-- ════════════════════════════════════════════════════════════════════════════
-- Lexical rules: token classes
-- ════════════════════════════════════════════════════════════════════════════

<identifier>    -> ('a'..'z' | 'A'..'Z' | '_') ('a'..'z' | 'A'..'Z' | '0'..'9' | '_')*
<integer>       -> ('0'..'9')+
<char>          -> ''' (any character except ''') '''          -- 'a'      ('''  is invalid)
<string>        -> '"' (any character except '"')* '"'         -- "Hello"  ("Hello"how" is invalid)
<whitespace>    -> (' ' | '\t' | '\v' | '\f')+                 -- discarded by the lexer
<comment>       -> '#' (any character except '\n')* '\n'       -- discarded by the lexer
                -> '{' (any character except '}')* '}'         -- may span lines; discarded

-- ════════════════════════════════════════════════════════════════════════════
-- Lexical rules: pre-defined tokens
-- ════════════════════════════════════════════════════════════════════════════

-- Keywords
'program'                                      -- start of the program
'const'  'type'  'var'                         -- declare constants, types, variables
'function'  'return'                           -- define a function / return from it
'begin'  'end'                                 -- start / end of a block
'output'  'read'                               -- print an expression or string / read input
'if'  'then'  'else'                           -- conditional
'while'  'do'                                  -- while loop
'repeat'  'until'                              -- repeat-until loop
'for'                                          -- for loop
'loop'  'pool'  'exit'                         -- loop-pool loop, and leaving it
'case'  'of'  'otherwise'                      -- case statement
'succ'  'pred'                                 -- successor / predecessor of an ordinal
'chr'  'ord'                                   -- character / ordinal conversion
'eof'                                          -- end of file

-- Operators
':='                                           -- assignment
':=:'                                          -- swap two variables
'='  '<>'  '<'  '<='  '>'  '>='                -- comparison
'+'  '-'  '*'  '/'  'mod'                      -- arithmetic
'and'  'or'  'not'                             -- logical

-- Punctuation
';'  ':'  ','  '.'  '..'  '('  ')'  '\n'

-- ════════════════════════════════════════════════════════════════════════════
-- Syntactic rules
-- ════════════════════════════════════════════════════════════════════════════

Winzig          -> 'program' Name ':' Consts Types Dclns SubProgs Body Name '.'  => "program";

Consts          -> 'const' Const list ',' ';'                                     => "consts"
                ->                                                                => "consts";

Const           -> Name '=' ConstValue                                            => "const";

ConstValue      -> '<integer>'
                -> '<char>'
                -> Name;

Types           -> 'type' (Type ';')+                                             => "types"
                ->                                                                => "types";

Type            -> Name '=' LitList                                               => "type";

LitList         -> '(' Name list ',' ')'                                          => "lit";

SubProgs        -> Fcn*                                                           => "subprogs";

Fcn             -> 'function' Name '(' Params ')' ':' Name
                   ';' Consts Types Dclns Body Name ';'                           => "fcn";

Params          -> Dcln list ';'                                                  => "params";

Dclns           -> 'var' (Dcln ';')+                                              => "dclns"
                ->                                                                => "dclns";

Dcln            -> Name list ',' ':' Name                                         => "var";

Body            -> 'begin' Statement list ';' 'end'                               => "block";

Statement       -> Assignment
                -> 'output' '(' OutExp list ',' ')'                               => "output"
                -> 'if' Expression 'then' Statement ('else' Statement)?           => "if"
                -> 'while' Expression 'do' Statement                              => "while"
                -> 'repeat' Statement list ';' 'until' Expression                 => "repeat"
                -> 'for' '(' ForStat ';' ForExp ';' ForStat ')' Statement         => "for"
                -> 'loop' Statement list ';' 'pool'                               => "loop"
                -> 'case' Expression 'of' Caseclauses OtherwiseClause 'end'       => "case"
                -> 'read' '(' Name list ',' ')'                                   => "read"
                -> 'exit'                                                         => "exit"
                -> 'return' Expression                                            => "return"
                -> Body
                ->                                                                => "<null>";

OutExp          -> Expression                                                     => "integer"
                -> StringNode                                                     => "string";

StringNode      -> '<string>';

Caseclauses     -> (Caseclause ';')+;

Caseclause      -> CaseExpression list ',' ':' Statement                          => "case_clause";

CaseExpression  -> ConstValue
                -> ConstValue '..' ConstValue                                     => "..";

OtherwiseClause -> 'otherwise' Statement                                          => "otherwise"
                -> ;

Assignment      -> Name ':=' Expression                                           => "assign"
                -> Name ':=:' Name                                                => "swap";

ForStat         -> Assignment
                ->                                                                => "<null>";

ForExp          -> Expression
                ->                                                                => "true";

Expression      -> Term
                -> Term '<=' Term                                                 => "<="
                -> Term '<' Term                                                  => "<"
                -> Term '>=' Term                                                 => ">="
                -> Term '>' Term                                                  => ">"
                -> Term '=' Term                                                  => "="
                -> Term '<>' Term                                                 => "<>";

Term            -> Factor
                -> Term '+' Factor                                                => "+"
                -> Term '-' Factor                                                => "-"
                -> Term 'or' Factor                                               => "or";

Factor          -> Factor '*' Primary                                             => "*"
                -> Factor '/' Primary                                             => "/"
                -> Factor 'and' Primary                                           => "and"
                -> Factor 'mod' Primary                                           => "mod"
                -> Primary;

Primary         -> '-' Primary                                                    => "-"
                -> '+' Primary
                -> 'not' Primary                                                  => "not"
                -> 'eof'
                -> Name
                -> '<integer>'
                -> '<char>'
                -> Name '(' Expression list ',' ')'                               => "call"
                -> '(' Expression ')'
                -> 'succ' '(' Expression ')'                                      => "succ"
                -> 'pred' '(' Expression ')'                                      => "pred"
                -> 'chr' '(' Expression ')'                                       => "chr"
                -> 'ord' '(' Expression ')'                                       => "ord";

Name            -> '<identifier>';
```

##### Railroad diagrams

The same rules are easier to follow as **railroad diagrams**. Start on the left, follow any track to the right, and every path you can take spells out something valid. Rounded green boxes are tokens; square blue boxes are other rules. A track that loops back means "repeat", and a track that bypasses a box means "optional".

**Statement**: each track is one kind of statement. The bottom track, with no boxes on it, is the empty statement.

<img src="assets/winzig-statement.svg" alt="Railroad diagram of the WinZig Statement rule" />

**Expression → Term → Factor → Primary**: these four rules are layered, and the layering is what sets operator precedence. `*` is handled in `Factor`, one level *below* `+` in `Term`, so in `a + b * c` the `b * c` is grouped first.

`Expression` compares at most two terms:

<img src="assets/winzig-expression.svg" alt="Railroad diagram of the WinZig Expression rule" />

`Term` is one or more factors joined by `+`, `-` or `or`:

<img src="assets/winzig-term.svg" alt="Railroad diagram of the WinZig Term rule" />

`Factor` is one or more primaries joined by `*`, `/`, `and` or `mod`:

<img src="assets/winzig-factor.svg" alt="Railroad diagram of the WinZig Factor rule" />

`Primary` is the smallest unit: a value, a name, a function call or a bracketed expression:

<img src="assets/winzig-primary.svg" alt="Railroad diagram of the WinZig Primary rule" />

> 💡 The grammar writes `Term` as `Term '+' Factor` (the rule refers to itself), but the diagram draws it as a loop. Both describe the same language: a chain of factors with operators between them. The recursive form is what tells the parser to group from the left, so `a - b - c` means `(a - b) - c`.

<details>
<summary><b>Example: parsing one line</b></summary>

A WinZigC program can't be just a statement: the `Winzig` rule requires a header, declarations and a `begin ... end` body. Here is a small but complete program:

```text
program Example:
var x, a, b : integer;
begin
    a := 1;
    b := 3;
    x := a + b * 2;
    output(x)
end Example.
```

The whole program goes through every step, but the trees for all of it would fill pages, so let's zoom in on the interesting line, `x := a + b * 2`.

**Step 1: lexical analysis.** The lexer turns the characters into a list of tokens. The spaces and the indentation are white space, so they are dropped. For our line, it produces:

| # | Text | Token |
|---|---|---|
| 1 | `x` | `<identifier>` |
| 2 | `:=` | `':='` |
| 3 | `a` | `<identifier>` |
| 4 | `+` | `'+'` |
| 5 | `b` | `<identifier>` |
| 6 | `*` | `'*'` |
| 7 | `2` | `<integer>` |

**Step 2: syntactic analysis.** The parser starts at the top rule, `Winzig`, and works its way down: it matches `program Example :`, the `var` declarations (`Dclns`), then the `Body`, which is a list of `Statement`s separated by `;`. The third statement is our line:

- `Statement` → `Assignment`, because the tokens start with a name followed by `:=`.
- `Assignment` → `Name ':=' Expression`, so `x` is the `Name` and `a + b * 2` must be an `Expression`.
- `Expression` → `Term`: there is no comparison operator (`<`, `=`, ...), so the whole right-hand side is one `Term`.
- `Term` → `Term '+' Factor`: `+` belongs to `Term`, so the parser splits at the `+`: `a` on the left, `b * 2` on the right.
- `b * 2` is a `Factor` → `Factor '*' Primary`, with `b` and `2` each a `Primary`.

Recording every rule it used gives the **parse tree** (also called the *concrete syntax tree*). Every token is in it, including `:=`:

```text
Statement
└── Assignment
    ├── Name
    │   └── <identifier> x
    ├── ':='
    └── Expression
        └── Term
            ├── Term
            │   └── Factor
            │       └── Primary
            │           └── Name
            │               └── <identifier> a
            ├── '+'
            └── Factor
                ├── Factor
                │   └── Primary
                │       └── Name
                │           └── <identifier> b
                ├── '*'
                └── Primary
                    └── <integer> 2
```

**Step 3: building the AST.** Most of the parse tree is scaffolding. The parser only creates a node when a rule has a `=> "name"` label: `Name ':=' Expression` builds `"assign"`, `Term '+' Factor` builds `"+"` and `Factor '*' Primary` builds `"*"`. Everything else (`Statement`, `Expression`, `Primary`, the `:=` token...) leaves no trace. What remains is the **abstract syntax tree**:

```text
assign
├── <identifier> x
└── +
    ├── <identifier> a
    └── *
        ├── <identifier> b
        └── <integer> 2
```

The tree's shape *is* the meaning: `*` sits below `+`, so `b * 2` is evaluated first and its result is added to `a`. No brackets were needed, because the grammar's layering (`Term` above `Factor`) put the nodes in the right place.

Python's own parser does exactly the same. Run `python3 -c "import ast; print(ast.dump(ast.parse('x = a + b * 2').body[0], indent=2))"` and you get the same tree, with Python's node names:

```text
Assign(
  targets=[
    Name(id='x', ctx=Store())],
  value=BinOp(
    left=Name(id='a', ctx=Load()),
    op=Add(),
    right=BinOp(
      left=Name(id='b', ctx=Load()),
      op=Mult(),
      right=Constant(value=2))))
```

> 💡 Treesitter builds this kind of tree for every buffer you open. Write `x = a + b * 2` in a Python file and run `:InspectTree` in Neovim: you'll see an `assignment` node whose right-hand side is a `binary_operator` containing another `binary_operator` for `b * 2`.

</details>

</details>


## TODO: Fill the rest